In [1]:
import pandas as pd
import numpy as np
from IPython.display import clear_output
from sklearn import metrics
from tensorflow.keras.utils import to_categorical
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data_path = '/content/drive/MyDrive/Ravaee/tcga_RSEM_Hugo_norm_count'

In [4]:
phenom_path = '/content/drive/MyDrive/Ravaee/TCGA_phenotype_denseDataOnlyDownload.tsv'

In [5]:
f = open(data_path)
samples = f.readline()
samples = samples.split('\t')

In [6]:
sample_count = len(samples[1:])

In [7]:
g = open(phenom_path)
g.readline()
_dict ={}
for line in g:
    fileds = line.split('\t')
    _dict[fileds[0].strip()] = fileds[3].strip()

In [8]:
phenotypes = []
for s in samples[1:]:
    if s.strip() not in _dict.keys():
        phenotypes.append('NA')
    else:
        phenotypes.append(_dict[s.strip()])

In [9]:
gen_count = 0
for line in f:
    fileds = line.split('\t')
    values = [float(x.strip()) for x in fileds[1:]]
#     
    values = np.asarray(values)
    if values.mean() >= 0.1:
        gen_count +=1
        clear_output(wait=True)
        print(gen_count)
f.close()

39158


In [10]:
data = np.zeros((gen_count , sample_count), dtype='float32')

In [11]:
f = open(data_path)
f.readline()
i = 0
for line in f:
    fileds = line.split('\t')
    values = [float(x.strip()) for x in fileds[1:]]
    values = np.asarray(values)
    if values.mean() >= 0.1:
        data[i] = values
        i += 1
        print(i)
        clear_output(wait=True)
f.close()

39158


In [12]:
data = data.T

In [13]:
x_train = data

In [14]:
x_train = x_train / np.max(x_train)

In [15]:
pca = PCA(n_components=1000,  svd_solver = 'full')

In [16]:
pca.fit(x_train)

PCA(n_components=1000, svd_solver='full')

In [17]:
reduced_data = pca.transform(x_train)

In [18]:
set_labels = set(phenotypes)

In [19]:
num_classes = len(set_labels)

In [20]:
dict_labels = {}
for i,l in enumerate(set_labels):
    dict_labels[l] = i

In [21]:
labels = []
for l in phenotypes:
    labels.append(dict_labels[l])

In [22]:
y_train = to_categorical(labels , num_classes=num_classes) 

In [23]:
idx = np.random.choice(x_train.shape[0], 1000, replace=False)
x_train = x_train[idx]
y_train = y_train[idx]

In [24]:
kmeans = KMeans(n_clusters=num_classes).fit(x_train)

In [25]:
y_hat = kmeans.labels_

print('ARI: ', metrics.adjusted_rand_score( np.argmax(y_train,axis=1), y_hat ))
print('AMI: ' ,metrics.adjusted_mutual_info_score( np.argmax(y_train,axis=1), y_hat ))
print('NMI: ',metrics.normalized_mutual_info_score( np.argmax(y_train,axis=1), y_hat ))
print('Mutal info: ',metrics.mutual_info_score( np.argmax(y_train,axis=1), y_hat ))

ARI:  0.5814204841713643
AMI:  0.7549535467968493
NMI:  0.7910638218008931
Mutal info:  2.587589413826338


In [26]:
from sklearn.cluster import AgglomerativeClustering

In [27]:
clustering = AgglomerativeClustering().fit(x_train)

In [28]:
y_hat = clustering.labels_

print('ARI: ', metrics.adjusted_rand_score( np.argmax(y_train,axis=1), y_hat ))
print('AMI: ' ,metrics.adjusted_mutual_info_score( np.argmax(y_train,axis=1), y_hat ))
print('NMI: ',metrics.normalized_mutual_info_score( np.argmax(y_train,axis=1), y_hat ))
print('Mutal info: ',metrics.mutual_info_score( np.argmax(y_train,axis=1), y_hat ))

ARI:  0.018684616284037864
AMI:  0.1568204713773796
NMI:  0.16522598917668463
Mutal info:  0.2884316778377953


In [29]:
from sklearn.cluster import DBSCAN

In [54]:
db = DBSCAN(eps=10, min_samples=1).fit(x_train)

In [55]:
y_hat = db.labels_

print('ARI: ', metrics.adjusted_rand_score( np.argmax(y_train,axis=1), y_hat ))
print('AMI: ' ,metrics.adjusted_mutual_info_score( np.argmax(y_train,axis=1), y_hat ))
print('NMI: ',metrics.normalized_mutual_info_score( np.argmax(y_train,axis=1), y_hat ))
print('Mutal info: ',metrics.mutual_info_score( np.argmax(y_train,axis=1), y_hat ))

ARI:  0.4438729896131364
AMI:  0.5266518635570115
NMI:  0.7189246512326349
Mutal info:  2.7972040143225994


In [56]:
from sklearn.mixture import GaussianMixture

In [57]:
gmm = GaussianMixture(n_components=5)

In [64]:
reduced_data = reduced_data[idx]

In [65]:
gmm.fit(reduced_data)

GaussianMixture(n_components=5)

In [66]:
gmm_labels = gmm.predict(reduced_data)

In [67]:
y_hat = gmm_labels

In [68]:


print('ARI: ', metrics.adjusted_rand_score( np.argmax(y_train,axis=1), y_hat ))
print('AMI: ' ,metrics.adjusted_mutual_info_score( np.argmax(y_train,axis=1), y_hat ))
print('NMI: ',metrics.normalized_mutual_info_score( np.argmax(y_train,axis=1), y_hat ))
print('Mutal info: ',metrics.mutual_info_score( np.argmax(y_train,axis=1), y_hat ))

ARI:  0.19148154931547093
AMI:  0.473824133780201
NMI:  0.489454632308929
Mutal info:  1.1446128979721137
